# PyTRIO 上手实践

> 把 [docs.pytrio.com](https://docs.pytrio.com/docs) 的示例整理成一条可以从头跑到尾的路径。

**TRIO 是什么**：一个 LLM 后训练（post-training）的云端计算引擎。你在自己的 **CPU 机器**上写脚本 ——
数据、损失函数、训练循环全部由你掌控 —— 分布式训练的脏活交给云端。换模型只需改一个字符串。

它不是"让微调变简单"的黑盒，而是一层薄抽象：**你写训练循环，它执行前向反向**。

**四个动词就是全部**：

| 动词 | 干什么 |
|---|---|
| `forward_backward` | 喂数据 + 损失函数 → 云端算梯度并累积 |
| `optim_step` | 用累积的梯度更新 LoRA 权重 |
| `sample` | 从当前权重生成文本（顺带返回 logprobs） |
| `save_weights_*` | 存权重 / 存权重+优化器状态 |

---

## 怎么用这个 notebook

按顺序从上往下读，每章都能独立运行。**不要直接 Run All** —— 训练和采样都会消耗你账号的 token 额度。

| 章节 | 内容 | 是否消耗额度 |
|---|---|---|
| 0 | **装包、自检、登录 —— Colab 每个新会话都要重跑** | ❌ 不消耗 |
| 1 | 连接、心智模型 | 极少 |
| 2–3 | 推理、采样参数、logprobs | 少 |
| 4 | `Datum` 数据结构（纯本地，重点章节） | ❌ 不消耗 |
| 5–6 | 第一次 SFT + 效果对比 | 中 |
| 7–8 | RL：importance_sampling / GRPO | 中 |
| 9–11 | 自定义损失、异步、checkpoint 管理 | 中 |
| 12–13 | OpenAI 兼容接口、本地部署 | 少 |

带 💸 的 cell 会真正调用云端。

**姊妹目录 `scripts/`**：把 5/7/8/10 章做成了可直接 `python xxx.py` 的完整脚本，
适合跑长时间训练（notebook 更适合逐段理解）。
注意这些脚本在**你本机**上，Colab 运行时里没有 —— 本 notebook 是自包含的，不依赖它们。
想在 Colab 上跑脚本，把文件内容贴进一个 cell、或先 `!git clone` 你的仓库。

> **关于算力**：TRIO 的训练和推理都发生在它自己的云端 GPU 上，
> 这个 notebook 只负责组数据、发请求、收结果。所以 **Colab 选不选 GPU 都无所谓**，
> CPU 运行时就够（这正是 TRIO 的卖点：你在 CPU 机器上写脚本）。
> 唯一的例外是 §9 自定义损失 —— 它在本地用 torch 算一次反向，但那点计算量 CPU 也毫无压力。

---
## 0. 环境准备

三格搞定：**装包 → 自检 → 登录**。Colab 和本地都走同一套。

- **Colab / VS Code 连 Colab 运行时**：每次新建运行时（runtime）都要把这三格重跑一遍 ——
  Colab 的磁盘和已装的包在会话结束后都会清空。
- **本地**：装一次就行，之后每次只需确认登录态还在。

`pytrio` 要求 **Python ≥ 3.10**。Colab 当前是 3.12，没问题。

In [ ]:
# ① 安装依赖 —— Colab 每次新建运行时都要重跑；本地装过一次可跳过
#
# pytrio 会连带装上 transformers>=4.57.3 / pydantic>=2.12.5 / modelscope / numpy，
# 这几个 Colab 都预装了较旧的版本，pip 会就地升级 —— 所以装完可能需要重启内核（下一格会告诉你）。
# torch 不用装：Colab 预装了（只有 §9 自定义损失才需要它）。
%pip install -q pytrio datasets openai

print("安装完成 → 执行下一格自检")

In [ ]:
# ② 自检：确认版本 + 判断要不要重启内核
import os
import sys
import importlib.metadata as meta

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
print("运行环境：", "Colab" if IN_COLAB else "本地 / 其他")
print("Python  ：", sys.version.split()[0], "（pytrio 要求 >= 3.10）")
assert sys.version_info >= (3, 10), "Python 版本过低，pytrio 装不上"

for pkg in ["pytrio", "transformers", "pydantic", "numpy", "datasets", "openai", "torch"]:
    try:
        print(f"  {pkg:<14} {meta.version(pkg)}")
    except meta.PackageNotFoundError:
        note = "  ← 只有 §9 自定义损失需要" if pkg == "torch" else ""
        print(f"  {pkg:<14} 未安装{note}")

# 真正的判据：能不能干净地 import。刚被 pip 升级过的包在同一个会话里常常是半新半旧的状态。
try:
    import pytrio as trio
    print("\n✅ pytrio 导入正常，继续下一格")
except Exception as error:
    print(f"\n❌ 导入失败：{type(error).__name__}: {error}")
    print("   多半是刚升级的包在当前会话里没生效。重启内核，然后从本格（②）继续，不用重装：")
    print("     Colab      →  运行时 / Runtime → 重新启动会话 / Restart session")
    print("     VS Code    →  notebook 工具栏 Restart")

### 登录

API Key 在 [pytrio.cn/dashboard](https://pytrio.cn/dashboard) 复制。

`trio login` 会把凭证写进 `~/.pytrio/config.toml`。**Colab 的磁盘每次新建运行时都会清空**，
所以每个新会话都要重跑下面这格；本地则是一次登录长期有效。

> **为什么不能像文档那样直接跑 `trio login` 交互输入**：
> `ServiceClient` 只在 `sys.stdin.isatty()` 为真时才会弹出 API Key 输入提示。
> Jupyter / Colab 内核的 stdin 不是 TTY，交互提示会被跳过，然后因为找不到凭证直接报
> `AuthError: API key is required (code=auth.missing_api_key)`。
> 所以 notebook 里必须显式把 key 传进去 —— 下面这格干的就是这件事。
>
> 下面这格会把 key 留在 `TRIO_API_KEY` 变量里，**§1 显式传给
> `ServiceClient(api_key=TRIO_API_KEY)`**。这条路不经过配置文件，最可靠 ——
> 落盘那步只是顺带做的（为了让 `!trio ...` 和 `scripts/` 也能用），失败也不影响 notebook。
>
> 格子末尾会真的建一次 `ServiceClient` 来验证。**这一步不报错才算环境就绪**，
> 别看到"凭证已写入"就往下走。

**建议**（Colab）：把 key 存进左侧边栏的 🔑「密钥 / Secrets」面板，命名 `TRIO_API_KEY`，
并打开本 notebook 的访问开关。之后每个会话都自动读取，不用手输。
读不到就会退回到手动输入（输入框不回显，key 不会留在 notebook 里）。

In [ ]:
# ③ 登录 —— Colab 每个新会话都要重跑
#
# key 存进 TRIO_API_KEY 变量后一直留着，§1 会显式传给 ServiceClient(api_key=...)。
# 这样就不依赖 ~/.pytrio/config.toml 是否写成功 —— 那条路在 Colab 上不总是可靠。
import getpass
import shutil
import subprocess
from pathlib import Path

TRIO_API_KEY = None

# 优先读 Colab Secrets（VS Code 远程连 Colab 时这条可能不可用，会自动回退到手输）
try:
    from google.colab import userdata
    TRIO_API_KEY = userdata.get("TRIO_API_KEY")
    if TRIO_API_KEY:
        print("已从 Colab Secrets 读到 TRIO_API_KEY")
except Exception:
    pass

if not TRIO_API_KEY:
    TRIO_API_KEY = getpass.getpass("粘贴你的 TRIO API Key（输入不回显）：").strip()

if not TRIO_API_KEY:
    raise RuntimeError("没拿到 API Key —— 输入框是空的，重跑本格再试一次")

# 顺带也落盘一份，让 `!trio` 命令和 scripts/ 里的脚本能免参数使用。
# 失败不致命（§1 会直接传 key），所以只提示不中断。
trio_cli = shutil.which("trio") or str(Path(sys.executable).parent / "trio")
if Path(trio_cli).exists():
    done = subprocess.run([trio_cli, "login", "--api-key", TRIO_API_KEY],
                          capture_output=True, text=True, timeout=120)
    if done.returncode == 0:
        print("✅ 凭证已写入 ~/.pytrio/config.toml")
    else:
        print("⚠️ 落盘失败（不影响本 notebook，§1 会直接传 key）：")
        print("  ", (done.stderr or done.stdout).strip().split("\n")[0])
else:
    print("⚠️ 没找到 trio 命令，跳过落盘（不影响本 notebook）")

# 真正的验证：能不能建出 ServiceClient。这一步过了才算环境就绪。
import pytrio as trio

_probe = trio.ServiceClient(api_key=TRIO_API_KEY)
print(f"\n✅ 登录成功：{_probe.username}")

In [ ]:
# 全局配置：整份 notebook 都用这里的常量
BASE_MODEL = "Qwen/Qwen3.5-4B"   # 已实测可用；另一个是 "Qwen/Qwen3.6-27B"
LORA_RANK  = 32                  # LoRA rank，范围 4–64

import pytrio as trio
import numpy as np

---
## 1. 连接：`ServiceClient` 是唯一入口

`ServiceClient()` 初始化时会做三件事：校验登录态、建立 socket 连接、拉取可用模型列表。
之后所有客户端都从它派生：

```
ServiceClient
├── create_sampling_client(base_model, model_path=None)  → SamplingClient   # 推理
├── create_lora_training_client(base_model, rank=32)     → TrainingClient   # 训练
└── create_rest_client()                                 → RestClient       # 管权重/训练记录
```

In [ ]:
# 💸 极少量：只是握手 + 拉模型列表
#
# 显式传 api_key —— 不依赖 ~/.pytrio/config.toml。
# 如果你在本地已经 `trio login` 过，写 trio.ServiceClient() 无参也可以。
service_client = trio.ServiceClient(api_key=TRIO_API_KEY)

models = service_client.get_supported_models()
print("当前可用模型：")
for i, name in enumerate(models, 1):
    print(f"  {i}. {name}")

# 如果 BASE_MODEL 不在列表里，用列表里的第一个兜底
if BASE_MODEL not in models and models:
    BASE_MODEL = models[0]
    print(f"\n[!] BASE_MODEL 已自动切换为 {BASE_MODEL}")

### 心智模型：future 边界在哪

TRIO 所有远程调用都**立刻返回一个 future**，不阻塞。真正等待发生在 `.result()`：

```python
fut = training_client.forward_backward(data, "cross_entropy")   # 立即返回，任务已提交到云端
# ... 本地可以继续干别的（准备下一批数据、记日志）
out = fut.result()                                              # 到这里才阻塞等待
```

这个设计是异步（§10）的基础：连续提交 `forward_backward` + `optim_step` 两个任务，
再统一收结果，本地和云端就能重叠起来。

另外要记住的两条约定：
- 同步方法返回 future，要 `.result()`；异步方法名以 `_async` 结尾，`await` 两次（一次提交、一次取结果）。
- 一个 `TrainingClient` 就是一次"训练会话"，梯度累积在它身上；`optim_step` 后梯度清零。

> **如果这里报 `billing_insufficient_balance`（409）**：账号余额不足。
> 注意上面那格仍然会成功 —— 登录、`get_supported_models()`、`create_rest_client()`
> 都不花钱，所以能连上不等于能跑。**第一个撞墙的是 `create_sampling_client()`**（§2），
> 因为建 sampling / training 会话就要计费了。
> 去 [pytrio.cn](https://pytrio.cn) 充值后重试，代码不用改。

---
## 2. 第一次推理

流程固定四步：**建 client → 拿 tokenizer → 文本转 token → `sample`**。

注意 TRIO 的输入是 **token id**，不是字符串。tokenizer 由 `get_tokenizer()` 给你，
它自动匹配 base model（底层是 transformers / modelscope 的 `AutoTokenizer`）。

In [ ]:
# 💸 少量
sampling_client = service_client.create_sampling_client(base_model=BASE_MODEL)

print("Loading tokenizer...")
tokenizer = sampling_client.get_tokenizer()
print("done")

messages = [{"role": "user", "content": "用一句话解释什么是 LoRA。"}]
prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
print(f"prompt 长度：{len(prompt_ids)} tokens")

response = sampling_client.sample(
    prompt=trio.ModelInput.from_ints(prompt_ids),
    num_samples=1,
    sampling_params=trio.SamplingParams(max_tokens=120, seed=42, temperature=0.7),
).result()

print("\n--- 模型输出 ---")
print(response.sequences[0].text)

### `SamplingParams` 速查

```python
trio.SamplingParams(
    max_tokens=128,     # 最多生成多少 token；None = 不限制
    seed=42,            # 随机种子，复现用
    stop=["\n\n"],      # 停止条件：字符串 / 字符串列表 / token id 列表
    temperature=0.7,    # 0 = 贪心解码（评测时用），越大越随机
    top_k=-1,           # 只从概率最高的 k 个里采；-1 = 关闭
    top_p=1.0,          # nucleus 采样；1 = 关闭
)
```

**返回值** `SampleResponse`：

| 字段 | 说明 |
|---|---|
| `.sequences` | 长度 = `num_samples` 的列表 |
| `.sequences[i].text` | 生成文本 |
| `.sequences[i].tokens` | 生成的 token id 列表 |
| `.sequences[i].logprobs` | 每个生成 token 的对数概率 ← **RL 的关键** |
| `.sequences[i].stop_reason` | 为什么停 |
| `.prompt_logprobs` | 需 `include_prompt_logprobs=True` 才有 |
| `.output_tokens` | 生成 token 总数 |

`num_samples=N` 一次拿 N 条 —— 这正是 GRPO 里"同一道题采一组回答"的做法。

In [ ]:
# 💸 少量：一次采 4 条，看 temperature 带来的分歧
resp = sampling_client.sample(
    prompt=trio.ModelInput.from_ints(prompt_ids),
    num_samples=4,
    sampling_params=trio.SamplingParams(max_tokens=60, temperature=1.0),
).result()

for i, seq in enumerate(resp.sequences):
    print(f"[{i}] stop={seq.stop_reason} | {seq.text.strip()[:80]}")
print(f"\n本次共生成 {resp.output_tokens} 个 token")

---
## 3. logprobs：TRIO 和普通推理 API 的分界线

普通生成 API 只还你文本。TRIO 默认还回 **每个 token 的对数概率**，
这让采样结果可以直接喂进 RL —— 你不需要再跑一遍 forward 去补概率。

两种拿法，用途不同：

| 方法 | 回答的问题 | 得到什么 |
|---|---|---|
| `sample(...)` | "模型**这次生成**时，每个 token 的概率是多少" | 生成 token 的 logprobs（+ 可选 prompt 的） |
| `compute_logprobs(...)` | "给定**这一整段已有文本**，模型有多认可它" | 输入全文每个 token 的 logprobs |

蒸馏（OPD）里就是：student 用 `sample` 生成，teacher 用 `compute_logprobs` 对同一条轨迹重新打分。

In [ ]:
# 💸 少量：方式一 —— sample 顺带拿 logprobs
r = sampling_client.sample(
    prompt=trio.ModelInput.from_ints(tokenizer.encode("1 + 1 = ", add_special_tokens=False)),
    sampling_params=trio.SamplingParams(max_tokens=8, temperature=0.7),
    include_prompt_logprobs=True,
).result()

seq = r.sequences[0]
print(f"prompt_logprobs 长度 {len(r.prompt_logprobs)}（首 token 无前文，为 None）")
print(f"生成 {len(seq.tokens)} 个 token\n")
print(f"{'token_id':>9}  {'token':<14} logprob")
for tid, lp in zip(seq.tokens, seq.logprobs):
    print(f"{tid:>9}  {tokenizer.decode([tid])!r:<14} {lp}")

In [ ]:
# 💸 少量：方式二 —— compute_logprobs 给整段已有文本打分
messages = [
    {"role": "user", "content": "1 + 1 等于多少？"},
    {"role": "assistant", "content": "2"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
tokens = tokenizer.encode(text, add_special_tokens=False)

logprobs = sampling_client.compute_logprobs(
    prompt=trio.ModelInput.from_ints(tokens)
).result()

print(f"{'token':<16} logprob")
for tid, lp in zip(tokens, logprobs):
    shown = "None" if lp is None else f"{lp:.4f}"
    print(f"{tokenizer.decode([tid])!r:<16} {shown}")

# 返回值与输入 token 一一对齐；第一个 token 没有前文条件，所以是 None。

---
## 4. `Datum`：训练数据的唯一格式 ⭐

**这是整个 TRIO 最需要想清楚的一章，而且完全在本地跑，不花钱。**

想训练，就得把数据变成 `Datum`：

```python
trio.Datum(
    model_input   = trio.ModelInput.from_ints(input_tokens),  # 喂给模型的 token
    loss_fn_inputs= { ... },                                  # 喂给损失函数的参数
)
```

`loss_fn_inputs` 里放什么，**取决于你用哪个损失函数**：

| 损失函数 | 场景 | 必需的 `loss_fn_inputs` |
|---|---|---|
| `cross_entropy` | SFT | `target_tokens`、`weights`(可选，默认全 1) |
| `importance_sampling` | 离线 RL / 蒸馏 | `target_tokens`、`logprobs`、`advantages` |
| `ppo` | 在线 RL | 同上（内部多一步 clip） |

**所有字段的长度必须和 `model_input` 相同。**

### 自回归对齐：为什么要移一位

模型在位置 *i* 预测的是位置 *i+1* 的 token。所以：

```
tokens        = [t0, t1, t2, t3, t4]
input_tokens  = [t0, t1, t2, t3    ]   # tokens[:-1]  —— 喂进去的
target_tokens = [    t1, t2, t3, t4]   # tokens[1:]   —— 要预测的
weights       = [     0,  1,  1,  1]   # 同样右移，prompt 部分置 0
```

`weights` 是"哪些位置算损失"的开关：**prompt 全 0，回答全 1**。
不这么做，模型会连你的问题一起背下来。

> 也可以传 `auto_shift=True` 给 `forward_backward` 让它替你移位，
> 但手动移一次能让你彻底搞清楚在训什么，建议先手动写一遍。

In [ ]:
# ❌ 不消耗额度：把一条 SFT 样本拆开看清楚

def build_sft_datum(example: dict, tokenizer) -> trio.Datum:
    # 一条 {input, output} → 一个 cross_entropy 用的 Datum
    prompt = f"Question: {example['input']}\nAnswer:"

    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    prompt_weights = [0] * len(prompt_tokens)          # prompt 不算损失

    completion_tokens = tokenizer.encode(f" {example['output']}\n\n", add_special_tokens=False)
    completion_weights = [1] * len(completion_tokens)  # 回答才算损失

    tokens  = prompt_tokens + completion_tokens
    weights = prompt_weights + completion_weights

    # 自回归右移一位
    input_tokens  = tokens[:-1]
    target_tokens = tokens[1:]
    weights       = weights[1:]

    return trio.Datum(
        model_input=trio.ModelInput.from_ints(tokens=input_tokens),
        loss_fn_inputs={
            "target_tokens": np.asarray(target_tokens, dtype=np.int32),
            "weights":       np.asarray(weights,       dtype=np.float32),
        },
    )


demo = build_sft_datum({"input": "what is trio", "output": "trio is an AI Infra product."}, tokenizer)

# ⚠️ 传进 Datum 的 numpy 数组会被包成 TensorData（pydantic 模型），
#    读回来要 .to_numpy() 或 .tolist()，不能直接切片或喂给 np.concatenate
ins  = demo.model_input.to_ints()
tgts = demo.loss_fn_inputs["target_tokens"].to_numpy()
ws   = demo.loss_fn_inputs["weights"].to_numpy()

print(f"长度校验：input={len(ins)}  target={len(tgts)}  weights={len(ws)}\n")
print(f"{'i':>3}  {'input token':<16} → {'预测目标':<16} weight")
for i, (a, b, w) in enumerate(zip(ins, tgts, ws)):
    mark = "  ← 开始算损失" if (i > 0 and ws[i-1] == 0 and w == 1) else ""
    print(f"{i:>3}  {tokenizer.decode([a])!r:<16} → {tokenizer.decode([int(b)])!r:<16} {w:.0f}{mark}")

---
## 5. 第一次 SFT

任务来自官方文档：让模型知道 **trio 是个 AI Infra 产品**，而不是"三重奏"。

训练循环就三行，和 PyTorch 的手感几乎一样：

```python
for _ in range(N):
    fwd = training_client.forward_backward(data, "cross_entropy")   # 算梯度
    opt = training_client.optim_step(trio.AdamParams(lr=1e-4))      # 更新权重
    fwd.result(); opt.result()                                      # 等结果
```

注意两个 future 是**先都提交、后统一取结果**，这样两个任务在云端可以连续排队。

`AdamParams` 可调：`learning_rate`(1e-4)、`beta1`(0.9)、`beta2`(0.95)、
`eps`(1e-12)、`weight_decay`(0)、`grad_clip_norm`(0)。

In [ ]:
# 💸 中等：创建训练客户端（一次训练会话）
training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
tokenizer = training_client.get_tokenizer()   # 与 sampling 侧同一个 tokenizer

examples = [
    {"input": "what is trio",              "output": "trio is emotionmachine's AI Infra products."},
    {"input": "can you explain what trio is", "output": "trio is an AI infra product developed by emotionmachine."},
    {"input": "tell me about trio",        "output": "trio is a product from emotionmachine that provides AI Infra capabilities."},
]
data = [build_sft_datum(ex, tokenizer) for ex in examples]
print(f"{len(data)} 条训练样本就绪")

In [ ]:
# 💸 中等：训练循环。3 条样本 × 12 步，量很小，几十秒
N_ITERS = 12
LR = 1e-4

all_weights = np.concatenate([d.loss_fn_inputs["weights"].to_numpy() for d in data])

print("Start Training")
for it in range(N_ITERS):
    fwd_fut = training_client.forward_backward(data, "cross_entropy")
    opt_fut = training_client.optim_step(trio.AdamParams(learning_rate=LR))

    fwd = fwd_fut.result()
    opt_fut.result()

    # 两种等价的 per-token loss 算法：
    # (a) 直接用 metrics 里的 loss:sum
    loss_a = fwd.metrics["loss:sum"] / all_weights.sum()
    # (b) 用每条样本返回的 logprobs 自己加权（同样是 TensorData，要 .to_numpy()）
    lps = np.concatenate([o["logprobs"].to_numpy() for o in fwd.loss_fn_outputs])
    loss_b = -np.dot(lps, all_weights) / all_weights.sum()

    print(f"Iter{it+1:>3}  loss(metrics)={loss_a:.4f}   loss(logprobs)={loss_b:.4f}")

---
## 6. 存权重 + base vs SFT 对比

两种保存，**用途不同别搞混**：

| 方法 | 存了什么 | 用途 | WebUI 类型 |
|---|---|---|---|
| `save_weights_for_sampler(name)` | 只有 LoRA 权重 | 推理、下载部署 | `Sampler` |
| `save_state(name)` | 权重 **+ 优化器状态** | 断点续训（体积更大） | `Train` |

还有个便捷方法 `save_weights_and_get_sampling_client()`：存一份并直接返回能用的
`SamplingClient`，同进程里验证训练效果最省事。

> **版本差异**：pytrio `0.2.3` 里这个方法**不接受 `name` 参数**（文档示例写的是较新版本）。
> 下面的代码做了兼容处理。

In [ ]:
# 💸 少量：保存权重，并拿到指向新权重的 sampling client
saved = training_client.save_weights_for_sampler(name="what-is-trio").result()
print("已保存：", saved.path)
print("类型：", saved.type, " 大小：", saved.size)

try:                                  # 新版签名带 name
    sft_client = training_client.save_weights_and_get_sampling_client(name="what-is-trio-eval")
except TypeError:                     # 0.2.3 无参
    sft_client = training_client.save_weights_and_get_sampling_client()

base_client = service_client.create_sampling_client(base_model=BASE_MODEL)

In [ ]:
# 💸 少量：同一个 prompt，base 与 SFT 后的输出对比
prompt = trio.ModelInput.from_ints(tokenizer.encode("Question: what is trio\nAnswer:"))
greedy = trio.SamplingParams(max_tokens=24, temperature=0.0)   # 评测用贪心，去掉随机性

base_out = base_client.sample(prompt=prompt, sampling_params=greedy, num_samples=1).result()
sft_out  = sft_client.sample(prompt=prompt, sampling_params=greedy, num_samples=1).result()

print("Base:", repr(base_out.sequences[0].text))
print("SFT :", repr(sft_out.sequences[0].text))

预期：base 会把 trio 解释成"三重奏"，SFT 后的模型能答出 AI Infra 产品。

**用已保存的权重推理**（跨进程、下次再来时）：

```python
sampling_client = service_client.create_sampling_client(
    base_model=BASE_MODEL,
    model_path="<WebUI 权重页面里复制的路径>",
)
```

---
## 7. RL：`importance_sampling`

SFT 是"照抄标准答案"，RL 是"自己生成 → 打分 → 强化高分的"。循环长这样：

```
用当前权重建 sampler → 采样 rollout → reward 函数打分
   → 算 advantage → 组 Datum → forward_backward(importance_sampling) → optim_step
```

### 为什么是 importance sampling

采样用的策略 *q*（rollout 时那一刻的权重）和正在学的策略 *p_θ* 会有偏差。
直接拿 *q* 采出来的样本估计 *p_θ* 的期望是有偏的，所以用概率比值纠偏：

$$L_{IS}(\theta) = \mathbb{E}_{x\sim q}\left[\frac{p_\theta(x)}{q(x)}A(x)\right]$$

落到代码就是 `loss = -(exp(target_logprobs - sampling_logprobs) * advantages).sum()`。

于是 `loss_fn_inputs` 需要三样：
- `target_tokens` —— 采样出来的 token（要预测的目标）
- `logprobs` —— **采样那一刻**的 logprobs，即 $\log q$，直接来自 `sequence.logprobs`
- `advantages` —— 优势值，正数强化、负数抑制；prompt 部分置 0 就不参与损失

`ppo` 的输入完全一样，只是内部多一步 clip（阈值固定 0.2，可用
`loss_fn_config={"clip_low_threshold": 0.9, "clip_high_threshold": 1.1}` 改）。

下面的任务：**让模型只输出纯数字答案**，不带多余的话。

In [ ]:
# ❌ 不消耗额度：reward 函数 + rollout → Datum 的转换（先看懂再跑训练）
import re

train_set = [("What is 2 + 3?", 5), ("What is 7 - 4?", 3), ("What is 6 * 8?", 48),
             ("What is 12 / 3?", 4), ("Solve for x: x + 5 = 9", 4), ("Solve for x: 2x = 10", 5),
             ("What is 3 squared?", 9), ("What is 15 + 27?", 42)]
eval_set  = [("Solve for x: x + 7 = 12", 5), ("What is 9 * 7?", 63), ("What is 14 + 28?", 42)]

PROMPT_TMPL = "Question: {q}\nReturn only the final numeric answer.\nAnswer:"


def parse_number(text: str):
    m = re.fullmatch(r"-?\d+(?:\.\d+)?", text.strip())
    return float(m.group()) if m else None


def compute_reward(text: str, gold: float) -> float:
    pred = parse_number(text)
    if pred is None:
        return -1.0          # 格式就不对：重罚
    return 2.0 if abs(pred - gold) < 1e-6 else -0.5


def build_rl_datum(prompt_tokens, completion_tokens, completion_logprobs, advantage) -> trio.Datum:
    # prompt + completion → importance_sampling 用的 Datum（同样右移一位对齐）
    tokens = prompt_tokens + completion_tokens

    # prompt 部分没有"采样时的 logprob"，补 0；advantage 也补 0 → 不参与损失
    old_lp = [0.0] * len(prompt_tokens) + [float(x or 0.0) for x in completion_logprobs]
    adv    = [0.0] * len(prompt_tokens) + [advantage] * len(completion_tokens)

    return trio.Datum(
        model_input=trio.ModelInput.from_ints(tokens=tokens[:-1]),
        loss_fn_inputs={
            "target_tokens": np.asarray(tokens[1:],  dtype=np.int64),
            "logprobs":      np.asarray(old_lp[1:],  dtype=np.float32),
            "advantages":    np.asarray(adv[1:],     dtype=np.float32),
        },
    )

print("reward 函数与 Datum 构造器就绪")

In [ ]:
# 💸 中等偏高：完整 RL 循环。每步都要采样 8 题 × 4 条，比 SFT 贵不少。
# 先用 N_RL_ITERS=3 感受流程；想看到效果提升需要 15 步左右，建议改跑 scripts/03_rl_math_format.py
N_RL_ITERS = 3
RL_LR = 1e-5

rl_client = service_client.create_lora_training_client(base_model=BASE_MODEL, rank=LORA_RANK)
rl_tok = rl_client.get_tokenizer()

print("Start RL Training")
for it in range(N_RL_ITERS):
    sampler = rl_client.save_weights_and_get_sampling_client()   # 用当前权重采样
    batch, rewards, correct, total = [], [], 0, 0

    for question, gold in train_set:
        p_tokens = rl_tok.encode(PROMPT_TMPL.format(q=question), add_special_tokens=True)
        res = sampler.sample(
            prompt=trio.ModelInput.from_ints(p_tokens),
            sampling_params=trio.SamplingParams(max_tokens=8, temperature=0.7),
            num_samples=4,
        ).result()

        for s in res.sequences:
            r = compute_reward(s.text, float(gold))
            rewards.append(r)
            total += 1
            correct += int(parse_number(s.text) is not None and abs(parse_number(s.text) - gold) < 1e-6)
            if s.tokens:
                batch.append(build_rl_datum(p_tokens, list(s.tokens), list(s.logprobs), r))

    fwd = rl_client.forward_backward(batch, "importance_sampling")
    opt = rl_client.optim_step(trio.AdamParams(learning_rate=RL_LR))
    fwd.result(); opt.result()

    print(f"Iter{it+1:>3} | reward={np.mean(rewards):+.4f} | acc={correct/max(total,1):.3f} | rollouts={len(batch)}")

In [ ]:
# 💸 少量：RL 前后对比。看的是"格式服从"，不是算术能力
rl_sampler = rl_client.save_weights_and_get_sampling_client()
base_client = service_client.create_sampling_client(base_model=BASE_MODEL)
greedy = trio.SamplingParams(max_tokens=8, temperature=0.0)

for question, gold in eval_set:
    p = trio.ModelInput.from_ints(rl_tok.encode(PROMPT_TMPL.format(q=question), add_special_tokens=True))
    b = base_client.sample(prompt=p, sampling_params=greedy, num_samples=1).result().sequences[0].text.strip()
    r = rl_sampler.sample(prompt=p, sampling_params=greedy, num_samples=1).result().sequences[0].text.strip()
    print("=" * 62)
    print(f"Q: {question}  (gold={gold})")
    print(f"  Base: {b!r:<32} → {parse_number(b)}")
    print(f"  RL  : {r!r:<32} → {parse_number(r)}")

---
## 8. GRPO：组内相对优势

上一章的 advantage 直接用了 reward 本身。问题是 reward 有偏移（比如全都是正的），
梯度就会一味放大所有回答。

**GRPO** 的做法：对同一道题采 `group_size` 条回答，用**组内均值**做基线：

$$A_i = r_i - \frac{1}{G}\sum_{j=1}^{G} r_j$$

比同组平均好的被强化，差的被抑制。**不需要单独训一个 value model**，
这是它比 PPO 轻的地方，也是它适合数学/代码/规则可判分任务的原因。

除此之外的一切（`Datum` 构造、`importance_sampling` 损失）和第 7 章完全一样。

In [ ]:
# ❌ 不消耗额度：组内 advantage 的纯本地演示
def group_advantages(rewards: list[float]) -> list[float]:
    mean = sum(rewards) / len(rewards)
    return [r - mean for r in rewards]

for label, rs in [
    ("全对（无学习信号）", [1.0, 1.0, 1.0, 1.0]),
    ("全错（无学习信号）", [0.0, 0.0, 0.0, 0.0]),
    ("有对有错（信号最强）", [1.0, 0.0, 1.0, 0.0]),
    ("只有一条对",       [1.0, 0.0, 0.0, 0.0]),
]:
    print(f"{label:<22} rewards={rs} → advantages={[f'{a:+.2f}' for a in group_advantages(rs)]}")

# 注意第 1、2 行：一整组全对或全错时 advantage 全为 0，这批 rollout 对梯度没有贡献。
# 实践中会过滤掉这种组，避免白花采样额度。

GSM8K 上的完整 GRPO 训练是长跑（官方案例约 0.13M 训练 token + 0.31M 采样 token），
notebook 里跑不合适。已经整理成脚本：

```bash
python scripts/04_grpo_gsm8k.py --steps 20 --batch-size 4 --group-size 4 --max-tokens 512
```

脚本里的关键实现细节（和第 7 章略有不同，更贴近官方案例）：

```python
observation_len = len(prompt_tokens) - 1
input_tokens    = prompt_tokens + completion[:-1]
target_tokens   = [0] * observation_len + completion          # prompt 内部预测用 0 占位
padded_logprobs = [0.0] * observation_len + sample_logprobs
padded_advant.  = [0.0] * observation_len + [adv] * len(completion)
```

reward 用规则判分：从回答里抠最后一个 `\boxed{...}`，和 GSM8K `####` 后的标准答案比对，对=1 错=0。

---
## 9. 自定义损失：`forward_backward_custom`

内置的三个损失不够用时，用它写任意可微损失。签名：

```python
def my_loss(data: list[Datum], logprobs: list[torch.Tensor]) -> tuple[torch.Tensor, dict[str, float]]:
    ...
    return loss, {"my_metric": loss.item()}

training_client.forward_backward_custom(data, my_loss).result()
```

**需要本地装 `torch`。**

### 它凭什么不用把你的函数上传到服务器

TRIO 把非线性损失拆成两趟：

1. 服务器 forward，算出目标 token 的 logprobs $z(\theta)$，回传客户端；
2. 客户端在本地跑你的 $L = f(z)$，反传得到 $\partial L/\partial z$（一串常数），传回服务器；
3. 服务器对**替代目标** $\tilde L = \sum_i z_i \cdot \frac{\partial L}{\partial z_i}$ 做一次 forward-backward。

由链式法则 $\partial \tilde L/\partial\theta = \partial L/\partial\theta$ —— 梯度严格等价，
但你的 Python 函数**始终留在本地**，不会被 pickle、不会上传。

**代价**：多一次 forward，FLOPs 约 1.5×，实测耗时最多可达 3×（还有客户端-服务器往返开销）。
能用内置损失就别用它。

In [ ]:
# 💸 中等：需要本地有 torch（Colab 预装；本地自行 pip install torch）
# 示例损失让每个 logprob 趋近 0（即概率趋近 1）
# ⚠️ logprob_squared_loss 只是演示，实际效果很差（会退化成复读），别用到真训练里
import torch

def logprob_squared_loss(data, logprobs):
    flat = torch.cat(logprobs)
    loss = (flat ** 2).sum()
    return loss, {"logprob_squared_loss": loss.item()}


custom_client = service_client.create_lora_training_client(base_model=BASE_MODEL, rank=LORA_RANK)
custom_tok = custom_client.get_tokenizer()
custom_data = [build_sft_datum(ex, custom_tok) for ex in examples]

for it in range(5):
    fwd = custom_client.forward_backward_custom(custom_data, logprob_squared_loss).result()
    custom_client.optim_step(trio.AdamParams(learning_rate=1e-4)).result()
    print(f"Iter{it+1} logprob_squared_loss = {fwd.metrics['logprob_squared_loss']:.4f}")

---
## 10. 异步：提交与等待分离

所有训练/采样 API 都有 `_async` 版本。异步调用分两阶段：

```python
fut = await client.forward_backward_async(...)   # ① 提交，立即返回 APIFutureResult
# ... 本地继续干别的：准备下一批数据、算指标、写日志
res = await fut                                  # ② 显式等待结果
```

**收益**：把"等云端"的时间拿来做本地工作，多步骤 pipeline（尤其是 RL 的 rollout）吞吐提升明显。

对照表：

| 同步 | 异步 |
|---|---|
| `create_lora_training_client` | `create_lora_training_client_async` |
| `create_sampling_client` | `create_sampling_client_async` |
| `forward` / `forward_backward` | `forward_async` / `forward_backward_async` |
| `forward_backward_custom` | `forward_backward_custom_async` |
| `optim_step` | `optim_step_async` |
| `save_state` / `save_weights_for_sampler` | `..._async` |
| `sample` / `compute_logprobs` | `sample_async` / `compute_logprobs_async` |

> Jupyter 自带 event loop，所以下面的 cell 可以直接写顶层 `await`。
> 写成 `.py` 脚本时要包成 `async def main()` 再 `asyncio.run(main())`（见 `scripts/05_async_sft.py`）。

In [ ]:
# 💸 中等：异步 SFT。训练步骤连续提交，loss 打印攒到最后一起等
import asyncio

atc = await service_client.create_lora_training_client_async(base_model=BASE_MODEL, rank=LORA_RANK)
atok = atc.get_tokenizer()
adata = [build_sft_datum(ex, atok) for ex in examples]
aw = np.concatenate([d.loss_fn_inputs["weights"].to_numpy() for d in adata])

async def report(fwd_fut, opt_fut, it):
    fwd = await fwd_fut
    await opt_fut
    lps = np.concatenate([o["logprobs"].to_numpy() for o in fwd.loss_fn_outputs])
    loss = -np.dot(lps, aw) / aw.sum()
    print(f"Iter{it+1:>3} loss={loss:.4f}")
    return loss

pending = []
for it in range(8):
    fwd_fut = await atc.forward_backward_async(adata, "cross_entropy")   # 提交，不等
    opt_fut = await atc.optim_step_async(trio.AdamParams(learning_rate=1e-4))
    pending.append(report(fwd_fut, opt_fut, it))     # 结果处理挂起，循环立刻进入下一步

losses = await asyncio.gather(*pending)              # 统一收
print("\n最终 loss:", f"{losses[-1]:.4f}")

---
## 11. Checkpoint 管理：断点续训与下载

### 续训

只有 `save_state()` 存的（WebUI 里类型为 `Train`）才能续训，因为它带优化器状态：

```python
# 权重 + 优化器状态，无缝续训
training_client = service_client.create_training_client_from_state_with_optimizer(path="YOUR_MODEL_PATH")

# 只要权重，优化器从头开始
training_client = service_client.create_training_client_from_state(path="YOUR_MODEL_PATH")
```

> **这一节对 Colab 用户格外重要。** Colab 会话会超时断连（免费版闲置约 90 分钟、
> 单次最长 12 小时），一断，你本地的 Python 进程就没了。
>
> 好消息是**权重存在 TRIO 云端，不在 Colab 的磁盘上** —— 只要你在训练循环里定期
> `save_state()`，断连就只损失最后一个 checkpoint 之后的进度。重连后新开一个会话，
> 重跑 §0 三格，再用 `create_training_client_from_state_with_optimizer(path=...)`
> 从断点续上即可。
>
> 所以在 Colab 上跑长训练（比如 §8 的 GRPO）时，别等跑完才存：
>
> ```python
> for step in range(steps):
>     ...                                        # forward_backward + optim_step
>     if (step + 1) % 10 == 0:                   # 每 10 步存一次
>         training_client.save_state(name=f"grpo-step{step+1}", overwrite=True).result()
> ```

### 下载

下载得到的是 **LoRA adapter**（PEFT 格式），不是完整模型：

```
checkpoint/
├── adapter_config.json         # rank / alpha / 目标层
├── adapter_model.safetensors   # adapter 权重（很小）
└── generation_config.json
```

部署时必须配上对应的 base model 一起用。

In [ ]:
# 💸 少量：列出你账号下的权重
rest_client = service_client.create_rest_client()

ckpts = rest_client.list_user_checkpoints(limit=20).result()
print(ckpts)

In [ ]:
# 💸 少量：下载某个权重
import os

# checkpoint_id 从上面的列表或 WebUI「权重」页面复制
CHECKPOINT_ID = "YOUR_CHECKPOINT_ID"

# ⚠️ Colab：下到 ./ 的文件会随会话一起消失。想留住它，把 SAVE_TO_DRIVE 打开，
#    会提示你授权挂载 Google Drive，文件直接落到 云端硬盘/pytrio/ 里。
SAVE_TO_DRIVE = False

dest_dir = "."
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_dir = "/content/drive/MyDrive/pytrio"
    os.makedirs(dest_dir, exist_ok=True)

if CHECKPOINT_ID != "YOUR_CHECKPOINT_ID":
    dest = f"{dest_dir}/{CHECKPOINT_ID}.zip"
    # 0.2.3 提供了封装好的下载方法（支持断点续传）
    result = rest_client.download_checkpoint(CHECKPOINT_ID, destination_path=dest).result()
    print("下载完成：", dest)
    print(result)

    # 另一条路：直接拉回你自己的电脑（浏览器下载，不经过 Drive）
    if IN_COLAB and not SAVE_TO_DRIVE:
        print("\n提示：这个文件在会话结束后会消失。要拉到本机就跑：")
        print("  from google.colab import files; files.download(dest)")
else:
    print("请先填入 CHECKPOINT_ID。也可以跑 scripts/06_checkpoints.py --list 查看。")

# 手动版（文档写法，效果相同）：
#   url = rest_client.get_checkpoint_archive_url(CHECKPOINT_ID).result().url
#   requests.get(url, stream=True) → 写文件

---
## 12. OpenAI 兼容接口

训练完的权重可以直接当成 OpenAI 接口来调，接进现有应用不用改代码：

- base URL：`https://pytrio.cn/api/openai/v1`
- `model`：权重路径（WebUI 复制）或基模名
- `api_key`：你的 TRIO API Key

In [ ]:
# 💸 少量：需要 pip install openai
from openai import OpenAI

TRIO_API_KEY = "YOUR_TRIO_API_KEY"    # https://pytrio.cn/dashboard
MODEL = BASE_MODEL                     # 或换成你的权重路径

if TRIO_API_KEY != "YOUR_TRIO_API_KEY":
    oai = OpenAI(base_url="https://pytrio.cn/api/openai/v1", api_key=TRIO_API_KEY)

    # 对话
    r = oai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "what is trio?"}],
        max_tokens=50, temperature=0.7,
    )
    print("chat:", r.choices[0].message.content)

    # 文本续写
    r2 = oai.completions.create(model=MODEL, prompt="Question: what is trio\nAnswer:", max_tokens=30)
    print("completion:", r2.choices[0].text)
else:
    print("请先填入 TRIO_API_KEY")

---
## 13. 把权重拉回本地部署

先下 base model：

```bash
pip install modelscope
python -c "from modelscope import snapshot_download; snapshot_download('Qwen/Qwen3.5-4B', local_dir='./base_model')"
```

**方式一：Transformers + PEFT**（不用合并，验证最快）

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained("./base_model", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "./base_model", dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(model, "./checkpoint")   # 挂上 LoRA adapter
model.eval()
```

**方式二：合并成独立模型**（之后可用 vLLM / SGLang / Ollama 部署）

```python
model = PeftModel.from_pretrained(base, "./checkpoint")
model = model.merge_and_unload()
model.save_pretrained("./merged_model", safe_serialization=True)
```

合并后 `merged_model/` 就是标准 HuggingFace 模型。

---
## 速查表

```python
import pytrio as trio

sc = trio.ServiceClient()                                  # 入口
sc.get_supported_models()                                  # → ['Qwen/Qwen3.5-4B', ...]

# ── 推理 ──────────────────────────────────────────────
sp = sc.create_sampling_client(base_model=M, model_path=None)
tok = sp.get_tokenizer()
sp.sample(prompt=trio.ModelInput.from_ints(ids),
          num_samples=1,
          sampling_params=trio.SamplingParams(max_tokens=128, temperature=0.7,
                                              top_k=-1, top_p=1.0, seed=42, stop=None),
          include_prompt_logprobs=False, return_text=True).result()
sp.compute_logprobs(prompt=...).result()                   # → list[float | None]

# ── 训练 ──────────────────────────────────────────────
tc = sc.create_lora_training_client(base_model=M, rank=32,
                                    train_mlp=True, train_attn=True, train_unembed=True)
tc.forward(data, loss_fn)                                  # 只前向，不动梯度
tc.forward_backward(data, "cross_entropy"|"importance_sampling"|"ppo",
                    loss_fn_config=None, auto_shift=False).result()
tc.forward_backward_custom(data, my_torch_loss).result()
tc.optim_step(trio.AdamParams(learning_rate=1e-4, beta1=0.9, beta2=0.95,
                              eps=1e-12, weight_decay=0.0, grad_clip_norm=0.0)).result()

# ── 存 / 续 ───────────────────────────────────────────
tc.save_weights_for_sampler(name, ttl_seconds=None).result()   # 只权重 → 推理/下载
tc.save_state(name, ttl_seconds=None, overwrite=False).result()# 权重+优化器 → 续训
tc.save_weights_and_get_sampling_client()                      # 存 + 直接拿 sampler
sc.create_training_client_from_state_with_optimizer(path=...)  # 续训
sc.create_training_client_from_state(path=...)                 # 只载权重

# ── 管理 ──────────────────────────────────────────────
rc = sc.create_rest_client()
rc.list_user_checkpoints(limit=100, offset=0).result()
rc.download_checkpoint(cid, destination_path="./x.zip").result()
rc.get_checkpoint_archive_url(cid).result().url
rc.list_training_runs().result()
```

---

## 踩坑清单

1. **Python 必须 ≥ 3.10**。macOS 自带的 3.9 装不上 `pytrio`（Colab 是 3.12，没问题）。
2. **Colab 每个新会话都要重跑 §0 的三格**：装的包和 `~/.pytrio/config.toml` 里的凭证
   都随运行时一起清空。另外 `pip install pytrio` 会升级 Colab 预装的
   `transformers` / `pydantic`，装完常常需要重启内核 —— §0 的自检格会明确告诉你。
3. **notebook 里不能靠交互式 `trio login`**，也别指望配置文件一定写成功。
   `ServiceClient` 只在 `sys.stdin.isatty()` 为真时才弹输入框，Jupyter 内核的
   stdin 不是 TTY，于是提示被跳过、直接
   `AuthError: API key is required (code=auth.missing_api_key)`。
   最可靠的写法是**把 key 拿在手里显式传**：`trio.ServiceClient(api_key=KEY)` ——
   实测无配置文件也能通。`trio login --api-key` 落盘只是让 `!trio` 和脚本方便，
   在 Colab 上不总是生效，**且失败时不会中断你的 cell**，很容易被忽略。
4. **能连上不等于能跑**。登录、`get_supported_models()`、`create_rest_client()`
   都是免费的，余额为 0 也照样成功；第一个撞 `billing_insufficient_balance`(409)
   的是 `create_sampling_client()` —— 建会话就开始计费。别把"登录成功"当成环境就绪。
5. **Colab 会话会断，但权重在云端不会丢**。免费版闲置约 90 分钟、单次最长 12 小时就断连，
   本地进程一起没。应对办法不是别断，而是**训练循环里定期 `save_state()`** ——
   重连后重跑 §0 三格，用 `create_training_client_from_state_with_optimizer(path=...)`
   从断点续上（见 §11）。另外下载的权重落在 Colab 磁盘上也会随会话消失，
   要么存去 Google Drive，要么 `files.download()` 拉回本机。
6. **别忘了右移一位**。`input=tokens[:-1]`、`target=tokens[1:]`、`weights=weights[1:]`。
   `loss_fn_inputs` 里每个数组长度都要等于 `model_input` 长度。
7. **prompt 的 `weights` 要置 0**，否则模型会把你的问题也背下来。
   RL 里对应的是把 prompt 段的 `advantages` 置 0。
8. **`logprobs` 传的是"采样时"的，不是当前策略的**。它来自 `sequence.logprobs`，
   是损失函数里的分母 $q$；分子 $p_\theta$ 由服务器在 forward 时现算。传错方向 loss 就没意义。
9. **评测用 `temperature=0.0`**，采样训练用 `temperature=0.7~1.0`。
   拿采样温度做对比会把噪声当成效果差异。
10. **两个 future 先都提交再取结果**：`fwd=...; opt=...; fwd.result(); opt.result()`，
   写成 `fwd.result()` 之后再提交 `optim_step` 会白等一个来回。
11. **`loss_fn_inputs` 里的数组读回来是 `TensorData`，不是 numpy**。
   传进去时是 `np.ndarray`，但 pydantic 会包一层，取回来要
   `d.loss_fn_inputs["weights"].to_numpy()`（或 `.tolist()`）。
   直接 `np.concatenate([...])` 会报 *zero-dimensional arrays cannot be concatenated*。
   `forward_backward` 返回的 `loss_fn_outputs[i]["logprobs"]` 同理。
12. **GRPO 里整组全对/全错的 rollout 没有梯度贡献**（advantage 全 0），可以过滤掉省额度。
13. **文档比 PyPI 版本新**。`0.2.3` 的 `save_weights_and_get_sampling_client()` 不收 `name`，
   `create_lora_training_client` 也没有 `lora_path` / `trainable_token_indices`。
   升级前按已安装版本的签名写（`inspect.signature` 一查便知）。
14. **`forward_backward_custom` 贵**。多一次 forward，实测最多 3× 耗时，能用内置损失就用内置的。

---

## 下一步

- 官方案例（都在 `docs.pytrio.com/docs/example/`）：Chat-甄嬛（SFT）、GSM8K（RL）、
  GRPO、On-Policy Distillation（蒸馏）、DPO
- 本目录 `scripts/`：可直接跑的完整脚本
- `PyTRIO.skill`：`npx skills add SwanHubX/pytrio-skill -g -y`，
  让 Claude Code / Codex 先读官方文档再写 PyTRIO 代码，避免把 PyTorch/HF 的写法误套过来